In [1]:
# ============================================================
# NB_Monitor_Logger
# Reusable Pipeline Monitoring Logger for Microsoft Fabric
# ------------------------------------------------------------
# PURPOSE : Log pipeline and activity run events to Delta table
# USAGE   : Call from any pipeline via Notebook activity
# INPUTS  : Parameters passed from pipeline
# OUTPUT  : Writes to monitoring.pipeline_run_log and pipeline_run_summary Delta tables
# Created by :Oumar Coulibaly
# Last updated by :Oumar Coulibaly

StatementMeta(, 5a7123ff-93c2-43b1-b849-fcc5f923697b, 3, Finished, Available, Finished, False)

In [2]:
# ============================================================
# variables setup 
# ============================================================


run_id = "RUN_20260408_102001"
pipeline_name = "01-PIP-LMS-raw to bronze_copy1"
workspace_name  = "'Laco_Playground'"
activity_name = "Copy data1"
activity_type = "Copy"
log_type = "activity"
start_time = "2026-04-08T10:20:06.122Z"
status = "Failed"
rows_read = "0"
rows_written = "0"
source_name = "0"
sink_name  = "target"
error_message = "The expression 'item().names' cannot be evaluated because property 'names' doesn't exist, available properties are 'name, type'."
error_code = "InvalidTemplate"
environment = "Production"
triggered_by = "Schedule"




StatementMeta(, 5a7123ff-93c2-43b1-b849-fcc5f923697b, 4, Finished, Available, Finished, False)

In [3]:
# ============================================================
# Printing variable value get from the piplene
# ============================================================

# Print the values for debugging (optional, remove in prod)
print("run_id:", run_id)
print("pipeline_name:", pipeline_name)
##print("pipeline_id:", pipeline_id)
print("workspace_name:", workspace_name)
print("activity_name:", activity_name)
print("activity_type:", activity_type)
print("log_type:", log_type)
print("start_time:", start_time)
print("status:", status)
print("rows_read:", rows_read)
print("rows_written:", rows_written)
print("source_name:", source_name)
print("sink_name:", sink_name)
print("error_message:", error_message)
print("error_code:", error_code)
print("environment:", environment)
print("triggered_by:", triggered_by)

StatementMeta(, 5a7123ff-93c2-43b1-b849-fcc5f923697b, 5, Finished, Available, Finished, False)

run_id: RUN_20260408_102001
pipeline_name: 01-PIP-LMS-raw to bronze_copy1
workspace_name: 'Laco_Playground'
activity_name: Copy data1
activity_type: Copy
log_type: activity
start_time: 2026-04-08T10:20:06.122Z
status: Failed
rows_read: 0
rows_written: 0
source_name: 0
sink_name: target
error_message: The expression 'item().names' cannot be evaluated because property 'names' doesn't exist, available properties are 'name, type'.
error_code: InvalidTemplate
environment: Production
triggered_by: Schedule


In [4]:
# ============================================================
# Preparing target tables
# ============================================================

MONITORING_DB    = "monitoring"
LOG_TABLE        = f"{MONITORING_DB}.pipeline_run_log"
SUMMARY_TABLE    = f"{MONITORING_DB}.pipeline_run_summary"

# ── Setup: Create monitoring DB and tables if not exists ──────
def setup_monitoring_tables():
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {MONITORING_DB}")

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {LOG_TABLE} (
            log_id              STRING,
            run_id              STRING,
            pipeline_id         STRING,
            pipeline_name       STRING,
            workspace_name      STRING,
            activity_name       STRING,
            activity_type       STRING,
            log_type            STRING,
            start_time          TIMESTAMP,
            end_time            TIMESTAMP,
            duration_seconds    BIGINT,
            status              STRING,
            error_message       STRING,
            error_code          STRING,
            rows_read           BIGINT,
            rows_written        BIGINT,
            source_name         STRING,
            sink_name           STRING,
            environment         STRING,
            triggered_by        STRING,
            log_date            DATE
        )
        USING DELTA
        PARTITIONED BY (log_date)
    """)

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {SUMMARY_TABLE} (
            run_id              STRING,
            pipeline_name       STRING,
            workspace_name      STRING,
            pipeline_start      TIMESTAMP,
            pipeline_end        TIMESTAMP,
            total_duration_sec  BIGINT,
            total_activities    INT,
            succeeded           INT,
            failed              INT,
            skipped             INT,
            total_rows_read     BIGINT,
            total_rows_written  BIGINT,
            overall_status      STRING,
            environment         STRING,
            triggered_by        STRING,
            log_date            DATE
        )
        USING DELTA
        PARTITIONED BY (log_date)
    """)

    print(f"✅ Monitoring tables ready in '{MONITORING_DB}' database")


StatementMeta(, 5a7123ff-93c2-43b1-b849-fcc5f923697b, 6, Finished, Available, Finished, False)

In [5]:
# ============================================================
# Loadind data into the monitoring.pipeline_run_log
# ============================================================



from datetime import datetime 
from pyspark.sql import Row 
from pyspark.sql.functions import col

# =========================
#  A Compute end_time & duration
# =========================

# Trim microseconds to 6 digits
#start_time_fixed = start_time[:26] + "Z" 
start_time_fixed = start_time[:26] 
start_dt = datetime.strptime(start_time_fixed, "%Y-%m-%dT%H:%M:%S.%fz")

end_dt = datetime.utcnow()
duration = int((end_dt - start_dt).total_seconds())



# =========================
# B Create DataFrame
# =========================
log_record  = [Row(
    run_id=run_id,
    pipeline_name=pipeline_name,
    workspace_name=workspace_name,
    activity_name=activity_name,
    activity_type=activity_type,
    log_type=log_type,
    start_time=start_dt,
    end_time=end_dt,
    duration=duration,
    status=status,
    rows_read=rows_read,
    rows_written=rows_written,
    source_name=source_name,
    sink_name=sink_name,
    error_message=error_message,
    error_code=error_code,
    environment=environment,
    triggered_by=triggered_by,
    log_date = datetime.utcnow().date()
)]

df = spark.createDataFrame(log_record)
df = df.withColumn("rows_read", col("rows_read").cast("long")) \
       .withColumn("rows_written", col("rows_written").cast("long"))

display(df)

# =========================
# C Write to existing table
# =========================
df.write \
    .mode("append") \
    .format("delta") \
    .option("mergeSchema", "true")\
    .saveAsTable(LOG_TABLE)




StatementMeta(, 5a7123ff-93c2-43b1-b849-fcc5f923697b, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 55279ee9-64cd-4a9c-9f3e-68ecb190ee0f)